In [2]:
# Data manipulation
import pandas as pd
import numpy as np

# SQL / SQLite
import sqlite3

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics
from scipy import stats

# Machine Learning
import sklearn
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# Excel / files
import openpyxl
import pyarrow as pa
import pyarrow.parquet as pq

# System / environment
import sys
import os

print("Environment ready")
print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("SQLite:", sqlite3.sqlite_version)
print("Scikit-learn:", sklearn.__version__)

Environment ready
Python: 3.11.17 | packaged by Anaconda, Inc. | (main, Oct  1 2026, 20:16:54) [MSC v.1942 64 bit (AMD64)]
Pandas: 3.0.6
NumPy: 2.4.6
SQLite: 3.53.4
Scikit-learn: 1.9.1


In [6]:
df_calendar = pd.read_excel("datasets/calendar_master.xlsx")
df_product = pd.read_excel("datasets/product_master.xlsx")
df_market = pd.read_excel("datasets/market_master.xlsx")
df_sales = pd.read_excel("datasets/daily_sales_accounts_2024_2026.xlsx")
df_rates = pd.read_excel("datasets/fx_rate_master_daily.xlsx")

conn = sqlite3.connect(":memory:")

df_calendar.to_sql("dim_calendar", conn, index=False, if_exists="replace")
df_product.to_sql("dim_product", conn, index=False, if_exists="replace")
df_market.to_sql("dim_market", conn, index=False, if_exists="replace")
df_sales.to_sql("fact_sales", conn, index=False, if_exists="replace")
df_rates.to_sql("fact_rates", conn, index=False, if_exists="replace")


5220

In [27]:
query = """
WITH sales_metrics AS (
    SELECT
        date AS sale_date,
        product_code,
        market_code,
        market_country,
        currency,
        CASE WHEN account = "sales" THEN value ELSE 0 END AS sales,
        CASE WHEN account = "accruals" THEN value ELSE 0 END AS accruals,
        CASE WHEN account = "units" THEN value ELSE 0 END AS units
    FROM fact_sales
),sales_summary AS (
SELECT
    sale_date,
    product_code,
    market_code,
    market_country,
    currency,
    SUM(sales) AS sales,
    SUM(accruals) AS accruals,
    CAST(SUM(units) AS INTEGER) AS units
FROM sales_metrics
GROUP BY sale_date, product_code, market_code, market_country, currency
),sales_with_rates AS (
    SELECT
        ss.sale_date,
        ss.product_code,
        ss.market_code,
        ss.market_country,
        ss.currency,
        ss.sales,
        ss.accruals,
        ss.units,
        fr.fx_rate_to_usd AS fx_rate,
        (ss.sales+ss.accruals) * fr.fx_rate_to_usd AS real_sales_usd
    FROM sales_summary ss
    LEFT JOIN fact_rates fr
        ON ss.sale_date = fr.rate_date AND ss.currency = fr.currency_code
), market_product_join AS (
    SELECT swr.*,
            dp.level_1_description AS medical_product,
            dp.level_2_description AS segment,
            dm.level_2_description AS region
    FROM sales_with_rates swr
    LEFT JOIN dim_product dp
        ON swr.product_code = dp.product_code
    LEFT JOIN dim_market dm
        ON swr.market_code = dm.level_3_code
)
SELECT * FROM market_product_join
"""

df_market_product_join = pd.read_sql_query(query, conn)
df_market_product_join.to_sql("fact_market_product_join", conn, index=False, if_exists="replace")


30000

In [37]:
query = """
SELECT market_country, region, sum(real_sales_usd) as total_sales_usd, SUM(units) as total_units
FROM fact_market_product_join
GROUP BY market_country, region
"""

pd.read_sql_query(query, conn)



,market_country,region,total_sales_usd,total_units
0,Argentina,Latin America,1.653821e+05,31724
1,Australia,Oceania,1.118916e+08,32208
2,Brazil,Latin America,3.089876e+07,32399
3,Canada,North America,1.270184e+08,33078
4,Chile,Latin America,1.787367e+05,31966
5,Colombia,,4.277463e+04,32737
6,France,Western Europe,1.991626e+08,34155
7,Germany,Western Europe,1.866922e+08,32777
8,Mexico,Latin America,9.244612e+06,32253
9,Spain,Southern Europe,1.845259e+08,32663


In [35]:
query = """
SELECT *
FROM fact_market_product_join
order by real_sales_usd asc
"""

pd.read_sql_query(query, conn)

,sale_date,product_code,market_code,market_country,currency,sales,accruals,units,fx_rate,real_sales_usd,medical_product,segment,region
0,2026-05-14 00:00:00,PRD_001,MKT_CO,Colombia,COP,96.67,-14.15,1,0.000254,0.020963,Orthopedic Products,Implants,
1,2025-10-21 00:00:00,PRD_193,MKT_CO,Colombia,COP,245.74,-42.53,3,0.000247,0.050286,Orthopedic Products,Implants,
2,2025-01-14 00:00:00,PRD_069,MKT_CO,Colombia,COP,248.00,-27.81,1,0.000249,0.054922,Orthopedic Products,Instruments,
3,2025-10-21 00:00:00,PRD_127,MKT_CO,Colombia,COP,311.87,-55.94,1,0.000247,0.063332,Trauma Products,Repair and Fixation,
4,2025-08-12 00:00:00,PRD_105,MKT_CO,Colombia,COP,325.79,-59.10,5,0.000247,0.065984,Orthopedic Products,Accessories,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,2025-07-01 00:00:00,PRD_199,MKT_GB,United Kingdom,GBP,342561.22,-59355.22,25,1.294961,366740.739126,Trauma Products,Instruments,Western Europe
29996,2026-02-04 00:00:00,PRD_068,MKT_GB,United Kingdom,GBP,314100.05,-15127.02,25,1.241592,371202.537212,Dental Products,Implants,Western Europe
29997,2026-06-24 00:00:00,PRD_068,MKT_GB,United Kingdom,GBP,312649.64,-19168.73,24,1.277181,374828.247984,Dental Products,Implants,Western Europe
29998,2026-01-20 00:00:00,PRD_070,MKT_GB,United Kingdom,GBP,295463.73,-7983.08,25,1.304313,374964.881285,Sports Medicine Products,Instruments,Western Europe
